In [84]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import os
from dotenv import load_dotenv

load_dotenv()

True

In [85]:
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")


In [86]:
query_com = """
SELECT DISTINCT v.Pedido,
    c.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2026-09-19'
    AND v.Fecha <= '2026-09-19'
    AND V.Cliente NOT IN ('C22222222222')
    AND v.Venta_Neta > 0
GROUP BY  v.Pedido,
    c.Cliente,
    c.Nombres + ' ' + c.Apellidos,
    c.Email,
    c.Celular,
    v.Fecha,
    v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_com, engine)

df_ventas.head()

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,103619,C1037634801,DANIELA LOPEZ ALVAREZ,danielalopez94d@gmail.com,3004013284,2026-09-19,Shopify,149363.32
1,103620,C1020776796,SULLY JAIMES,sjaimesg@unal.edu.co,3006157906,2026-09-19,Shopify,172051.42
2,103621,C39681439,LINA WILLIAMSON,linawilliamson@hotmail.com,3102108024,2026-09-19,Shopify,87601.27
3,103622,C32296342,JULIANA VELASQUEZ,julyvela@hotmail.com,3137686971,2026-09-19,Shopify,519935.63
4,103623,C52809454,ANGELA ESCOBAR,angieescobar@hotmail.com,3213438788,2026-09-19,Shopify,334229.33


In [87]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Pedido,
# 	v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta,
#     Canal,
#     SUM(Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'SEPTIEMBRE_2026')
# 	AND Fecha BETWEEN '2026-09-05' AND '2026-09-30'
# GROUP BY  v.Pedido,
# 	v.Cliente,
#     cc.Nombres + ' ' +  cc.Apellidos,
#     cc.Email,
#     cc.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [88]:
# Leer archivo
df_indigitall = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_indigitall[['email', 'campaignName', 'sentAt', 'openedAt', 'clicks']].copy()

# Renombrar columnas
df_campania.rename(columns={
    'email': 'Email',
    'campaignName': 'campania',
    'sentAt': 'fecha_envio',
    'openedAt': 'apertura',
    'clicks': 'clicks'
}, inplace=True)

# 1. Convertir la columna openedAt de string a lista real de fechas
df_campania['apertura'] = df_campania['apertura'].astype(str)
df_campania['apertura'] = df_campania['apertura'].apply(
    lambda x: ast.literal_eval(x) if x.startswith("[") else []
)

# 2. Expandir: cada fecha en una fila
df_expanded = df_campania.explode('apertura').copy()

# 3. Limpiar y convertir a datetime
df_expanded['apertura'] = df_expanded['apertura'].str.strip()
df_expanded['apertura'] = pd.to_datetime(
    df_expanded['apertura'],
    format='%Y-%m-%dT%H:%M:%S.%fZ',
    utc=True,
    errors='coerce'
).dt.date

#Convertir la columna clicks en Fecha
df_campania['clicks'] = df_campania['clicks'].astype(str)

def extraer_fecha_click(valor):
    try:
        # Convertir string a objeto Python (lista de dicts)
        lista = ast.literal_eval(valor)
        if isinstance(lista, list) and len(lista) > 0 and 'clicked_at' in lista[0]:
            fecha = pd.to_datetime(lista[0]['clicked_at'], utc=True, errors='coerce')
            return fecha.date()  # solo fecha
    except Exception:
        return pd.NaT
    return pd.NaT

df_campania['clicks'] = df_campania['clicks'].apply(extraer_fecha_click)

# 4. Normalizar correos
df_expanded['Email'] = df_expanded['Email'].str.lower().str.strip()

# 5. Elegir solo la primera apertura por email
df_abiertos = df_expanded.dropna(subset=['apertura']) \
    .sort_values('apertura') \
    .drop_duplicates(subset='Email', keep='first')

df_clics =  df_campania[df_campania['clicks'].notnull()].copy()

# Filtro correos abiertos hasta una fecha maxima
fecha_maxima = pd.to_datetime('2026-09-19').date()
df_abiertos_filtrados = df_abiertos[df_abiertos['apertura'] <= fecha_maxima]
df_clicks_filtrados = df_clics[df_clics['clicks'] <= fecha_maxima]
total_abiertos = df_abiertos_filtrados['Email'].nunique()
total_clicks = df_clicks_filtrados['Email'].nunique()

In [89]:
# Normalizar df_ventas
df_ventas['Email'] = df_ventas['Email'].str.lower().str.strip()
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_merge = pd.merge(df_ventas, df_abiertos, on='Email', how='inner')

# Filtrar ventas dentro de 5 días desde la apertura
df_merge['dias_post_apertura'] = (df_merge['Fecha_Venta'] - df_merge['apertura']).apply(lambda x: x.days)
df_atribucion = df_merge[
    (df_merge['dias_post_apertura'] >= 0) &
    (df_merge['dias_post_apertura'] <= 5)
]

# Sumar la venta total atribuida
tottal_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Número de clientes que abrieron hasta {fecha_maxima}: {total_abiertos}")
print(f"Número de clientes que hicieron clics hasta {fecha_maxima}: {total_clicks}")
print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {tottal_venta:,.2f}"  )
# Resultado final
df_atribucion.head()

Número de clientes que abrieron hasta 2026-09-19: 11566
Número de clientes que hicieron clics hasta 2026-09-19: 206
Total de Ventas: 71
Total de venta atribuida: 19,843,086.22


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,campania,fecha_envio,apertura,clicks,dias_post_apertura
0,103632,C1032373905,DIEGO ARIAS URREA,diego.arias22@outlook.com,3132098920,2026-09-19,Shopify,80668.80,ULT HORAS SURA,2026-09-19T14:36:05.000Z,2026-09-19,[],0
1,103644,C52647250,BIBIANA HOYOS,bibihoyos10@gmail.com,3132458469,2026-09-19,Shopify,248938.88,ULT HORAS SURA,2026-09-19T14:37:04.000Z,2026-09-19,[],0
2,103645,C38668064,MARIA FERNANDA OLIVEROS,fernanda.oliveros@icloud.com,3114519291,2026-09-19,Shopify,770765.17,ULT HORAS SURA,2026-09-19T14:35:12.000Z,2026-09-19,[],0
3,103657,C31864862,GEMMA ELSY SALAZAR,gemma.salazar@hotmail.com,3117478699,2026-09-19,Shopify,189067.50,ULT HORAS SURA,2026-09-19T14:35:21.000Z,2026-09-19,"[{'clicked_at': '2026-09-19T14:54:18.000Z', 'c...",0
4,103658,C41501435,OLGA CECILIA CASTELLANOS LUQUE,olgacaslu@hotmail.com,3188269407,2026-09-19,Shopify,173942.10,ULT HORAS SURA,2026-09-19T14:35:46.000Z,2026-09-19,[],0


In [90]:
df_atribucion.to_excel('Ventas.xlsx', index=False)